# Activity 1: Prediction — Smart PH Crop Recommender 🌾
**STIWNU Computing Seminar • November 23, 2026**  
**Colab Runtime:** Default **CPU** (save your free GPU for Activity 2) &nbsp;•&nbsp; **Time:** ~25 minutes

### What is *prediction* in Machine Learning?
Instead of writing rules by hand (*"if rainfall > 200 mm, plant rice"*), we show the computer **many past examples** and let it **learn the pattern** by itself. Then it can **predict** the answer for new, unseen cases.

```
 Inputs (features)                                   Output (prediction)
 soil N, P, K, temperature,  ──►  🤖 trained model  ──►  "Plant RICE (92% sure)"
 humidity, pH, rainfall
```

### What you will do
| Step | What happens | Key concept |
|---|---|---|
| 1–2 | Load the data & look at the first 2 rows | Features vs. label |
| 3 | Check data quality & visualize | Exploratory Data Analysis (EDA) |
| 4 | Split into training and test sets | Never test on data the model has seen |
| 5 | Train a Decision Tree & see its rules | How a model "thinks" |
| 6 | Compare 3 models | Cross-validation |
| 7 | Test the winner on unseen data | Accuracy, confusion matrix |
| 8 | Find which factors matter most | Feature importance |
| 9 | Make **your own** prediction | Inference |
| 10 | Save the model & launch a web app | Deployment |

> **How to use this notebook:** Run each cell **from top to bottom** with `Shift + Enter`. Read the text cell *before* each code cell; it explains what's about to happen.
>
> 💬 **Think**: a question to discuss with your seatmate &nbsp;•&nbsp; 🧪 **Try it**: change a value and re-run &nbsp;•&nbsp; 💡 **Key idea**: the takeaway to remember

In [ ]:
# ==============================================================================
# SETUP: Install & import libraries (takes ~20 seconds)
# ==============================================================================
!pip install -q kagglehub gradio

import os, glob, joblib, kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

sns.set_theme(style='whitegrid')
print('✅ All libraries imported!')

## Step 1: Load the Dataset
We use the public **Crop Recommendation Dataset**: **2,200 farm samples** across **22 crops** (rice, maize, banana, coconut, coffee, mango, and more).

| Column | Meaning | Unit |
|---|---|---|
| `N`, `P`, `K` | Nitrogen, Phosphorus, Potassium in the soil | ratio (kg/ha) |
| `temperature` | Average temperature | °C |
| `humidity` | Relative humidity | % |
| `ph` | Soil acidity (7 = neutral) | 0–14 |
| `rainfall` | Rainfall | mm |
| `label` | ✅ **The crop best suited to these conditions** (what we want to predict) | |

> If the venue Wi-Fi blocks Kaggle, the cell automatically creates a similar practice dataset so you can keep going.

In [ ]:
# ==============================================================================
# STEP 1: Download the dataset (with an automatic offline fallback)
# ==============================================================================
try:
    dataset_dir = kagglehub.dataset_download('atharvaingle/crop-recommendation-dataset')
    df = pd.read_csv(glob.glob(os.path.join(dataset_dir, '*.csv'))[0])
    print('✅ Loaded the real Kaggle dataset')
except Exception as e:
    print(f'⚠️ Could not reach Kaggle ({e}). Generating a practice dataset instead...')
    rng = np.random.default_rng(42)
    profiles = {  # crop: (N, P, K, temperature, humidity, ph, rainfall)
        'rice': (80, 48, 40, 24, 82, 6.4, 230), 'maize': (78, 48, 20, 22, 65, 6.2, 85),
        'banana': (100, 82, 50, 27, 80, 6.0, 105), 'coconut': (22, 17, 30, 27, 95, 6.0, 175),
        'coffee': (101, 29, 30, 25, 59, 6.8, 158), 'mango': (20, 27, 30, 31, 50, 5.8, 95),
    }
    noise = (10, 6, 5, 2, 5, 0.4, 25)
    rows = [[max(0, m + rng.normal(0, s)) for m, s in zip(p, noise)] + [crop]
            for crop, p in profiles.items() for _ in range(100)]
    df = pd.DataFrame(rows, columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label'])

print(f'Dataset size: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## Step 2: Meet the Data — Look at the First 2 Samples
Every **row** is one farm observation. The first 7 columns are the **features** (inputs, called `X`), and `label` is the **target** (the answer, called `y`).

Let's read the first 2 rows like a human would.

In [ ]:
# ==============================================================================
# STEP 2: Read the first 2 rows in plain English
# ==============================================================================
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

for i in range(2):
    row = df.iloc[i]
    print(f'🌱 Sample #{i + 1}')
    print(f'   Soil nutrients : N={row.N:.0f}, P={row.P:.0f}, K={row.K:.0f}')
    print(f'   Climate        : {row.temperature:.1f} °C, {row.humidity:.0f}% humidity, {row.rainfall:.0f} mm rain')
    print(f'   Soil pH        : {row.ph:.2f}')
    print(f'   ✅ Best crop    : {row.label.upper()}\n')

print(f'The dataset contains {df["label"].nunique()} different crops:')
print(', '.join(sorted(df['label'].unique())))

## Step 3: Exploratory Data Analysis (EDA)
**Rule #1 of ML: look at your data before training anything.** We check for:
1. **Missing values**: empty cells that can crash training
2. **Duplicates**: repeated rows that inflate scores
3. **Class balance**: does every crop have a similar number of samples?

In [ ]:
# ==============================================================================
# STEP 3a: Data quality checks
# ==============================================================================
print('Missing values per column:')
print(df.isnull().sum().to_string(), '\n')
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

plt.figure(figsize=(12, 3.5))
df['label'].value_counts().plot(kind='bar', color='#2E7D32')
plt.title('Number of samples per crop (balanced = all bars the same height)')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

Now the interesting part: **do different crops really need different conditions?** If they do, a model can learn to tell them apart.

In [ ]:
# ==============================================================================
# STEP 3b: How do conditions differ per crop?
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = df.groupby('label')['rainfall'].median().sort_values().index
sns.boxplot(data=df, x='label', y='rainfall', order=order, ax=axes[0], color='#81C784')
axes[0].set_title('Rainfall needed per crop')
axes[0].tick_params(axis='x', rotation=90)

some_crops = [c for c in ['rice', 'coconut', 'coffee', 'mango', 'banana', 'maize'] if c in df['label'].unique()]
sns.scatterplot(data=df[df['label'].isin(some_crops)], x='temperature', y='humidity',
                hue='label', ax=axes[1], s=40, alpha=0.8)
axes[1].set_title('Temperature vs. humidity for selected crops')
plt.tight_layout()
plt.show()

💬 **Think:**
- Which crop needs the **most** rainfall? Which needs the **least**?
- In the scatter plot, do crops form separate "clusters"? Could you draw lines to separate them by hand?

💡 **Key idea:** A model can only predict well if the features actually carry information about the label. Separate clusters mean the job is learnable.

## Step 4: Split the Data into Training and Test Sets
Think of a student preparing for a board exam:
- 📘 **Training set (80%)**: the practice questions the model studies from.
- 📝 **Test set (20%)**: the **final exam**, kept hidden until the very end.

If we tested on questions the model already saw, we would only measure **memorization**, not real learning. `stratify=y` keeps the same mix of crops in both sets.

In [ ]:
# ==============================================================================
# STEP 4: Separate features (X) and target (y), then split 80/20
# ==============================================================================
X = df[FEATURES]   # inputs
y = df['label']    # answers

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # 20% hidden for the final exam
    random_state=42,    # same split every time (reproducible)
    stratify=y          # same crop proportions in both sets
)
print(f'📘 Training samples: {len(X_train)}')
print(f'📝 Test samples    : {len(X_test)}  (hidden until Step 7)')

## Step 5: Train Your First Model, a Decision Tree 🌳
A **Decision Tree** learns a flowchart of *yes/no questions* like **"Is rainfall ≤ 150 mm?"**. It's the easiest model to understand because you can **see its rules**.

We'll train a tiny tree (only 3 questions deep) so it fits on screen.

In [ ]:
# ==============================================================================
# STEP 5: Train a small Decision Tree and visualize its rules
# ==============================================================================
small_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
small_tree.fit(X_train, y_train)          # <-- this one line is the "learning"!

plt.figure(figsize=(20, 8))
plot_tree(small_tree, feature_names=FEATURES, class_names=small_tree.classes_,
          filled=True, rounded=True, fontsize=9, impurity=False, proportion=True)
plt.title('The rules learned by a 3-level Decision Tree')
plt.show()

print(f'Accuracy of this tiny tree on the training data: {small_tree.score(X_train, y_train):.1%}')

💬 **Think:** Which feature did the tree ask about **first** (top box)? That's the single most useful question for separating crops.

🧪 **Try it:** Change `max_depth=3` to `max_depth=6` and re-run. Does accuracy improve? (The picture becomes hard to read, which is the trade-off between **accuracy** and **explainability**.)

## Step 6: Compare 3 Models with Cross-Validation
Which algorithm should we choose? We compare 3 popular ones **using only the training data**:

| Model | Idea |
|---|---|
| **Decision Tree** | One flowchart of yes/no questions |
| **Logistic Regression** | Draws straight-line boundaries between classes |
| **Random Forest** | 🌳🌳🌳 Hundreds of trees that **vote** together |

**5-fold cross-validation** splits the training set into 5 parts, trains on 4 and checks on the 5th, repeated 5 times. This gives a fair score without touching the test set.

> `Pipeline` + `StandardScaler` puts all features on a similar scale (rainfall in the hundreds, pH around 6) *inside* each fold, so no information leaks from the validation part.

In [ ]:
# ==============================================================================
# STEP 6: Compare models with 5-fold cross-validation
# ==============================================================================
models = {
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Logistic Regression': Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=2000))]),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f'{name:20s} → average accuracy {scores.mean():.2%}  (± {scores.std():.2%})')

pd.Series(results).sort_values().plot(kind='barh', color=['#A5D6A7', '#66BB6A', '#1B5E20'], figsize=(8, 3))
plt.xlim(0.8, 1.0)
plt.title('Cross-validation accuracy (higher is better)')
plt.show()

best_name = max(results, key=results.get)
best_model = models[best_name].fit(X_train, y_train)   # retrain the winner on ALL training data
print(f'\n🏆 Winner: {best_name}. Retrained on the full training set.')

## Step 7: The Final Exam: Evaluate on the Unseen Test Set
Now, and **only now**, we unlock the test set. Let's first see predictions for a few individual samples, then the overall score.

In [ ]:
# ==============================================================================
# STEP 7a: Predictions vs. actual answers for the first 10 test samples
# ==============================================================================
y_pred = best_model.predict(X_test)

preview = X_test.head(10).copy()
preview['actual crop'] = y_test.head(10).values
preview['predicted crop'] = y_pred[:10]
preview['correct?'] = np.where(preview['actual crop'] == preview['predicted crop'], '✅', '❌')
preview

In [ ]:
# ==============================================================================
# STEP 7b: Overall test performance
# ==============================================================================
print(f'🎯 TEST ACCURACY: {accuracy_score(y_test, y_pred):.2%}  '
      f'({(y_test == y_pred).sum()} of {len(y_test)} correct)\n')
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Greens',
                                        xticks_rotation=90, colorbar=False)
ax.set_title('Confusion matrix: the diagonal = correct predictions')
plt.tight_layout()
plt.show()

**How to read the results:**
- **Accuracy**: % of all predictions that were correct.
- **Precision** (per crop): *"When the model says rice, how often is it really rice?"*
- **Recall** (per crop): *"Of all the real rice samples, how many did the model find?"*
- **Confusion matrix**: numbers **off the diagonal** are mistakes. Which crops get confused with each other, and why might that be?

## Step 8: Why Did the Model Decide That? (Feature Importance)
A good AI system should be **explainable**. Random Forest tells us how much each feature helped its decisions.

In [ ]:
# ==============================================================================
# STEP 8: Which soil/climate factors matter most?
# ==============================================================================
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

importance.plot(kind='barh', color='#2E7D32', figsize=(8, 4))
plt.title('Feature importance (Random Forest)')
plt.xlabel('Importance (higher = more useful for predicting the crop)')
plt.tight_layout()
plt.show()

🧪 **Try it: an experiment.** What happens if a farmer has **no rain gauge**? Remove `'rainfall'` from the list below and re-run. How much does accuracy drop? Try removing other features too.

In [ ]:
# ==============================================================================
# EXPERIMENT: Train with fewer features
# ==============================================================================
FEATURES_TO_USE = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']   # 👈 remove some!

exp_model = RandomForestClassifier(n_estimators=200, random_state=42)
exp_model.fit(X_train[FEATURES_TO_USE], y_train)
exp_acc = exp_model.score(X_test[FEATURES_TO_USE], y_test)
print(f'Features used: {FEATURES_TO_USE}')
print(f'Test accuracy: {exp_acc:.2%}')

## Step 9: Make Your Own Prediction 🔮
This is what a deployed model does all day: **take new inputs → return a prediction**.

🧪 **Try it:** Move the sliders on the right (in Colab), then run the cell. Can you find conditions where the model recommends **coffee**? **Coconut**? **Mango**?

In [ ]:
# ==============================================================================
# STEP 9: Predict the best crop for YOUR farm (use the sliders in Colab →)
# ==============================================================================
N           = 85    #@param {type:"slider", min:0, max:140, step:1}
P           = 48    #@param {type:"slider", min:5, max:145, step:1}
K           = 40    #@param {type:"slider", min:5, max:205, step:1}
temperature = 24.5  #@param {type:"slider", min:8, max:45, step:0.5}
humidity    = 82    #@param {type:"slider", min:14, max:100, step:1}
ph          = 6.4   #@param {type:"slider", min:3.5, max:9.5, step:0.1}
rainfall    = 230   #@param {type:"slider", min:20, max:300, step:5}

my_farm = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=FEATURES)
probs = pd.Series(best_model.predict_proba(my_farm)[0], index=best_model.classes_).sort_values(ascending=False)

print(f'🌾 Recommended crop: {probs.index[0].upper()} ({probs.iloc[0]:.0%} confidence)\n')
probs.head(5)[::-1].plot(kind='barh', color='#43A047', figsize=(7, 3))
plt.title('Top 5 crop recommendations')
plt.xlabel('Model confidence')
plt.show()

## Step 10 (Bonus): Save the Model & Launch a Web App 🚀
A trained model is just a file. We save it with `joblib`, load it back (like a real web server would), and wrap it in a **Gradio** web app that anyone can use from a browser.

> If the venue network blocks the public link, change `share=True` to `share=False`.

In [ ]:
# ==============================================================================
# STEP 10: Save → reload → serve as a web app
# ==============================================================================
import gradio as gr

joblib.dump(best_model, 'crop_recommender.joblib')
print(f"💾 Saved model: {os.path.getsize('crop_recommender.joblib') / 1024:.0f} KB")
loaded_model = joblib.load('crop_recommender.joblib')

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    x = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=FEATURES)
    return {c.title(): float(p) for c, p in zip(loaded_model.classes_, loaded_model.predict_proba(x)[0])}

gr.Interface(
    fn=recommend_crop,
    inputs=[gr.Slider(0, 140, 85, label='Nitrogen (N)'), gr.Slider(5, 145, 48, label='Phosphorus (P)'),
            gr.Slider(5, 205, 40, label='Potassium (K)'), gr.Slider(8, 45, 24.5, label='Temperature (°C)'),
            gr.Slider(14, 100, 82, label='Humidity (%)'), gr.Slider(3.5, 9.5, 6.4, label='Soil pH'),
            gr.Slider(20, 300, 230, label='Rainfall (mm)')],
    outputs=gr.Label(num_top_classes=3, label='Top 3 recommended crops'),
    title='🌾 Smart PH Crop Recommender',
    examples=[[85, 48, 40, 24.5, 82, 6.4, 230], [22, 18, 30, 27.5, 94, 6.0, 175], [102, 28, 30, 25.5, 58, 6.8, 160]],
).launch(share=True)

## ✅ Recap
1. **Prediction** = learning patterns from labeled examples, then applying them to new data.
2. **Always explore your data first**: quality and patterns.
3. **Keep a hidden test set** so you measure real learning, not memorization.
4. **Compare models fairly** with cross-validation; simple models are easier to explain.
5. **Look beyond accuracy**: confusion matrix, precision/recall, feature importance.

### 💬 Check your understanding
1. Why can't we evaluate a model on the same data it was trained on?
2. If a model has 99% accuracy but always confuses *rice* and *jute*, would you trust it for a farmer growing jute?
3. Name one **other** prediction problem in your community that this same workflow could solve.

➡️ **Next: Activity 2: Image Processing**, where the inputs are **photos** instead of numbers.